# <center>Modeling
This notebook will primarily contain the modeling algorithms used on the Student dataset. The dataset lends itself to regression and classification tasks, making it versatile for both avenues. 
## Outline:

1. Modeling objectives:
- Regression Task =   
    - **Can we predict a student's final grade?**  
- Classification Task =   
    - **Can we classify students according to their final academic performance?**  
    - **Can we predict whether a student will pass or fail?** 

2. Data preparation
    - Feature/target separation
    - Train/test split
    - Numerical features
    - Categorical features
    - Encoding
    - Preprocessing pipeline
    - Leakage considerations

3. Regression

    - 3.1 Regression objective
    - 3.2 Target: Final Grade (G3)
    - 3.3 Baseline model
    - 3.4 Linear Regression
    - 3.5 Decision Tree Regressor
    - 3.6 Random Forest Regressor
    - 3.7 Model evaluation
    - 3.8 Model comparison
    - 3.9 Regression findings

4. Classification

    - 4.1 Classification objective
    - 4.2 Creating Pass/Fail target
    - 4.3 Class distribution
    - 4.4 Baseline model
    - 4.5 Logistic Regression
    - 4.6 Decision Tree Classifier
    - 4.7 Random Forest Classifier
    - 4.8 Model evaluation
    - 4.9 Model comparison
    - 4.10 Classification findings

6. Overall Modeling Conclusions

7. Limitations

8. Recommendations / Future Work

In [10]:
# specify libraries to use
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# modeling libraries
# Baseline model
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Split the data
from sklearn.model_selection import train_test_split, GridSearchCV

# Metrics
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# Decision Tree model
from sklearn.tree import DecisionTreeClassifier

# Random Forest model
from sklearn.ensemble import RandomForestClassifier

In [2]:
# load the dataset
math_df = pd.read_csv('student-mat.csv', sep = ';')
port_df = pd.read_csv('student-por.csv', sep = ';')

In [3]:
print('Maths Dataframe:')
display(math_df.head())

print('Portuguese Dataframe:')
display(port_df.head())

Maths Dataframe:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


Portuguese Dataframe:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [4]:
# Open and read the txt file
with open('student.txt', 'r', encoding='utf-8') as file:
    content = file.read()

# Print the text to the VS Code terminal
print(content)


# Attributes for both student-mat.csv (Math course) and student-por.csv (Portuguese language course) datasets:
1 school - student's school (binary: "GP" - Gabriel Pereira or "MS" - Mousinho da Silveira)
2 sex - student's sex (binary: "F" - female or "M" - male)
3 age - student's age (numeric: from 15 to 22)
4 address - student's home address type (binary: "U" - urban or "R" - rural)
5 famsize - family size (binary: "LE3" - less or equal to 3 or "GT3" - greater than 3)
6 Pstatus - parent's cohabitation status (binary: "T" - living together or "A" - apart)
7 Medu - mother's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
8 Fedu - father's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
9 Mjob - mother's job (nominal: "teacher", "health" care related, civil "services" (e.g. administrative or police), "at_home" or 

## Regression Task
### Maths Subject

In [5]:
math_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  higher    

In [6]:
math_df.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3'],
      dtype='object')

### Dummy Regressor 
**What if we completely ignored student characteristics and just predicted the average grade for everyone?**    
- It essentially predicts the training-set mean for every student
- It doesn't look at age, study time, failures, absences, parental education, etc.
- gives a reference point for judging whether the actual regression model has learned anything useful

One nuance: because the Dummy Regressor ignores the predictors, the preprocessing isn't actually necessary for this particular model. But keeping the same pipeline structure is useful because you'll replace DummyRegressor with LinearRegression, DecisionTreeRegressor, etc. later. It keeps your workflow consistent and prevents you from accidentally changing preprocessing between models.

In [ ]:
# Define predictors and target variable
X = math_df.drop(columns=['G1', 'G2', 'G3']) 
# G1 and G2 were excluded from the predictors because they are previous-period grades and are highly predictive of the final grade (G3). 
# Excluding them allows the model to investigate whether student demographic, academic, family, and behavioral characteristics can predict final performance without relying directly on prior grades.
y = math_df['G3']

# Outline numerical and categorical variables
numerical_features = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime',
       'failures', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences']
categorical_features = ['school', 'sex', 'address', 'famsize', 'Pstatus', 
       'Mjob', 'Fjob', 'reason', 'guardian',  'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic']

# preprocessing
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(
        drop = 'first',
        handle_unknown = 'ignore'
    ))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

dummy_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy='mean'))
])

dummy_model.fit(X_train, y_train)

dummy_y_pred = dummy_model.predict(X_test)


In [ ]:
# model the data
math_baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()) # G3 is a numeric final grade (0–20)
])

# fit the model
math_baseline_model.fit(X_train, y_train)

# make predictions
math_baseline_y_pred = math_baseline_model.predict(X_test)
